# Pronóstico del precio de Tesla con Series de Tiempo Difusas (Fuzzy Time Series)

**Materia:** Inteligencia Computacional — UPTC

**Integrantes:** Andrés Santiago Daza Quintana, Óscar Alejandro Roa Donzel

**Dataset:** Tesla Stock Price History (Kaggle) — columnas Date, Open, High, Low, Close, Adj Close, Volume.

**Objetivo hipotético:** pronosticar el precio de cierre ajustado (Adj Close) de la acción de Tesla para el día siguiente, a partir de su comportamiento histórico reciente.

**Algoritmo de inteligencia computacional:** Series de Tiempo Difusas (Fuzzy Time Series), basado en Song & Chissom (1993) y Chen (1996). El modelo se implementa "a mano", sin librerías especializadas de FTS.

In [ ]:
# Importamos las librerías que vamos a usar en todo el ejercicio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## 1. Carga de datos

In [ ]:
# 1. Carga de datos

# Leemos el archivo CSV de Tesla (la columna 0 del archivo es un índice que no necesitamos)
df = pd.read_csv('TESLA.csv', index_col=0)

# Convertimos la columna Date a formato de fecha (el archivo viene en formato mes/dia/año)
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y')

# Ordenamos cronológicamente por si el archivo no viene ordenado
df = df.sort_values('Date').reset_index(drop=True)

# Nos quedamos solo con la fecha y el precio de cierre ajustado (nuestra serie objetivo)
serie = df[['Date', 'Adj Close']].copy()

print("Cantidad de registros:", len(serie))
print(serie.head())
print(serie.tail())

## 2. Análisis exploratorio de la serie de tiempo

In [ ]:
# 2.1 Graficamos la serie completa del precio de cierre ajustado
plt.figure(figsize=(12, 5))
plt.plot(serie['Date'], serie['Adj Close'])
plt.title('Precio de cierre ajustado de Tesla (2010-2024)')
plt.xlabel('Fecha')
plt.ylabel('Adj Close (USD)')
plt.grid(True)
plt.show()

In [ ]:
# 2.2 Calculamos los retornos diarios (variación porcentual día a día)
serie['Retorno'] = serie['Adj Close'].pct_change()

# Graficamos los retornos diarios
plt.figure(figsize=(12, 4))
plt.plot(serie['Date'], serie['Retorno'])
plt.title('Retornos diarios de Tesla')
plt.xlabel('Fecha')
plt.ylabel('Retorno diario')
plt.grid(True)
plt.show()

In [ ]:
# 2.3 Prueba de estacionariedad de Dickey-Fuller (ADF) sobre el precio en niveles
resultado_adf = adfuller(serie['Adj Close'])

print("Estadístico ADF:", resultado_adf[0])
print("p-valor:", resultado_adf[1])

if resultado_adf[1] < 0.05:
    print("Con un 95% de confianza, la serie es estacionaria")
else:
    print("Con un 95% de confianza, la serie NO es estacionaria (tiene tendencia)")

In [ ]:
# 2.4 Graficamos la autocorrelación (ACF) y la autocorrelación parcial (PACF)
fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(serie['Adj Close'], lags=40, ax=ejes[0])
ejes[0].set_title('ACF - Adj Close')
plot_pacf(serie['Adj Close'], lags=40, ax=ejes[1])
ejes[1].set_title('PACF - Adj Close')
plt.show()

# Nota: la ACF decae muy lentamente (no llega rápido a cero), lo cual es típico de series
# no estacionarias con tendencia, y coincide con el resultado de la prueba de Dickey-Fuller.

## 3. Partición cronológica en entrenamiento y prueba

In [ ]:
# 3. Partición cronológica (NO aleatoria, NO estratificada)
# Usamos el 15% de las fechas más recientes como test, y el resto como train

n = len(serie)
porcentaje_test = 0.15

n_test = int(n * porcentaje_test)
n_train = n - n_test

train = serie.iloc[:n_train].reset_index(drop=True)
test = serie.iloc[n_train:].reset_index(drop=True)

print("Registros de entrenamiento:", len(train), "->", train['Date'].min().date(), "a", train['Date'].max().date())
print("Registros de prueba:", len(test), "->", test['Date'].min().date(), "a", test['Date'].max().date())

## 4. Modelo baseline: naive forecast y regresión lineal simple

In [ ]:
# Para que la comparación entre modelos sea justa, todos van a pronosticar el precio de "hoy"
# usando únicamente el precio real de "ayer" (pronóstico a un paso hacia adelante)

valores_completos = serie['Adj Close'].values

# valor_anterior_test[i] es el precio real del día justo anterior a test.iloc[i]
valor_anterior_test = valores_completos[n_train - 1 : n - 1]

# valor_real_test[i] es el precio real que queremos pronosticar
valor_real_test = valores_completos[n_train : n]

In [ ]:
# 4.1 Baseline naive: el pronóstico de mañana es igual al precio de hoy
pronostico_naive = valor_anterior_test.copy()

In [ ]:
# 4.2 Baseline de regresión lineal simple: precio_hoy = pendiente * precio_ayer + intercepto
# La ajustamos SOLO con los datos de entrenamiento
x_train = train['Adj Close'].values[:-1]   # precio de "ayer"
y_train = train['Adj Close'].values[1:]    # precio de "hoy"

pendiente, intercepto = np.polyfit(x_train, y_train, 1)
print("Pendiente:", pendiente, "- Intercepto:", intercepto)

pronostico_regresion = pendiente * valor_anterior_test + intercepto

## 5. Modelo de Series de Tiempo Difusas (Fuzzy Time Series)

Seguimos, de forma simplificada, los pasos clásicos de Song & Chissom (1993) y Chen (1996):

1. Definir el universo del discurso a partir del train.
2. Partir el universo en intervalos y asociar un conjunto difuso a cada intervalo.
3. Fuzzificar la serie (asignar cada precio a su intervalo).
4. Generar las relaciones lógicas difusas (FLR): A_i -> A_j.
5. Agrupar las FLR en grupos (FLRG).
6. Usar los FLRG para pronosticar (desfuzzificar con el punto medio de los intervalos).

In [ ]:
# 5.1 Definimos el universo del discurso a partir del train (con un pequeño margen extra)
n_intervalos = 10  # cantidad de intervalos (conjuntos difusos) en los que dividimos el universo

margen = (train['Adj Close'].max() - train['Adj Close'].min()) * 0.05
U_min = train['Adj Close'].min() - margen
U_max = train['Adj Close'].max() + margen

ancho_intervalo = (U_max - U_min) / n_intervalos

# Límites de cada intervalo u_1, u_2, ..., u_n_intervalos
limites = [U_min + i * ancho_intervalo for i in range(n_intervalos + 1)]

# Punto medio de cada intervalo (lo usamos después para "desfuzzificar" el pronóstico)
puntos_medios = [(limites[i] + limites[i + 1]) / 2 for i in range(n_intervalos)]

print("Universo del discurso: [%.2f, %.2f]" % (U_min, U_max))
print("Ancho de cada intervalo: %.2f" % ancho_intervalo)

In [ ]:
# 5.2 Fuzzificación: a cada intervalo u_i le asociamos un conjunto difuso A_i
# Simplificamos asignando cada precio real al intervalo (conjunto difuso) al que pertenece

def fuzzificar_valor(valor):
    indice = int((valor - U_min) / ancho_intervalo)
    # Si el valor cae justo en el límite superior o se sale un poco del universo, lo recortamos
    if indice < 0:
        indice = 0
    if indice >= n_intervalos:
        indice = n_intervalos - 1
    return indice

# Fuzzificamos toda la serie de entrenamiento: cada precio queda etiquetado como A_0, A_1, ..., A_9
etiquetas_train = [fuzzificar_valor(v) for v in train['Adj Close'].values]

print("Primeras 10 etiquetas difusas del train:", etiquetas_train[:10])

In [ ]:
# 5.3 Generamos las relaciones lógicas difusas (FLR): A_i -> A_j
# Comparamos la etiqueta de cada día con la etiqueta del día siguiente

relaciones = []
for i in range(len(etiquetas_train) - 1):
    origen = etiquetas_train[i]
    destino = etiquetas_train[i + 1]
    relaciones.append((origen, destino))

print("Cantidad de relaciones (FLR) generadas:", len(relaciones))
print("Ejemplos de relaciones (origen, destino):", relaciones[:5])

In [ ]:
# 5.4 Agrupamos las FLR en grupos (FLRG): todas las relaciones que parten del mismo A_i

flrg = {}
for origen, destino in relaciones:
    if origen not in flrg:
        flrg[origen] = []
    if destino not in flrg[origen]:
        flrg[origen].append(destino)

# Mostramos los grupos obtenidos, por ejemplo A_3 -> A_2, A_3, A_4
for origen in sorted(flrg.keys()):
    destinos_texto = ["A_%d" % d for d in flrg[origen]]
    print("A_%d -> %s" % (origen, destinos_texto))

In [ ]:
# 5.5 Función de pronóstico del modelo difuso (regla de Chen, 1996)
# Para pronosticar el día t: fuzzificamos el precio real del día t-1 y buscamos su FLRG

def pronosticar_fts(valor_anterior):
    etiqueta = fuzzificar_valor(valor_anterior)
    if etiqueta in flrg and len(flrg[etiqueta]) > 0:
        # Si el grupo tiene uno o varios destinos, el pronóstico es el promedio de sus puntos medios
        destinos = flrg[etiqueta]
        pronostico = np.mean([puntos_medios[d] for d in destinos])
    else:
        # Si esa etiqueta nunca apareció como origen en el train, usamos el punto medio de su propio intervalo
        pronostico = puntos_medios[etiqueta]
    return pronostico

# 5.6 Aplicamos el pronóstico difuso a todo el conjunto de test
pronostico_fts = np.array([pronosticar_fts(v) for v in valor_anterior_test])

## 6. Evaluación final

In [ ]:
# 6.1 Funciones de error: MAE, RMSE y MAPE

def calcular_mae(real, pronostico):
    return np.mean(np.abs(real - pronostico))

def calcular_rmse(real, pronostico):
    return np.sqrt(np.mean((real - pronostico) ** 2))

def calcular_mape(real, pronostico):
    return np.mean(np.abs((real - pronostico) / real)) * 100

In [ ]:
# 6.2 Calculamos las métricas para cada modelo y las juntamos en una tabla
resultados = pd.DataFrame({
    'Modelo': ['Naive', 'Regresión lineal', 'Fuzzy Time Series'],
    'MAE': [
        calcular_mae(valor_real_test, pronostico_naive),
        calcular_mae(valor_real_test, pronostico_regresion),
        calcular_mae(valor_real_test, pronostico_fts),
    ],
    'RMSE': [
        calcular_rmse(valor_real_test, pronostico_naive),
        calcular_rmse(valor_real_test, pronostico_regresion),
        calcular_rmse(valor_real_test, pronostico_fts),
    ],
    'MAPE (%)': [
        calcular_mape(valor_real_test, pronostico_naive),
        calcular_mape(valor_real_test, pronostico_regresion),
        calcular_mape(valor_real_test, pronostico_fts),
    ],
})

print(resultados)

In [ ]:
# 6.3 Gráfico comparativo: precio real vs. pronósticos de cada modelo en el conjunto de test
plt.figure(figsize=(14, 6))
plt.plot(test['Date'], valor_real_test, label='Precio real', linewidth=2)
plt.plot(test['Date'], pronostico_naive, label='Naive', linestyle='--')
plt.plot(test['Date'], pronostico_regresion, label='Regresión lineal', linestyle='--')
plt.plot(test['Date'], pronostico_fts, label='Fuzzy Time Series', linestyle='--')
plt.title('Precio real vs. pronósticos en el conjunto de test')
plt.xlabel('Fecha')
plt.ylabel('Adj Close (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 7. Conclusión
#
# El baseline naive y la regresión lineal simple obtienen errores muy similares y bajos (MAPE ~2.7%),
# porque el precio de una acción se comporta casi como un "camino aleatorio": el mejor predictor del
# precio de mañana termina siendo el precio de hoy. El modelo de Series de Tiempo Difusas obtiene un
# error notablemente mayor (MAPE ~6%), ya que al agrupar el precio en solo 10 intervalos discretos
# pierde precisión frente al comportamiento continuo de la serie, mostrando que en este caso no logra
# superar a un baseline simple.